# 06. Pandas: czyszczenie danych

Czas: ok. 25 min

W surowym pliku wszystko jest tekstem, a gminy, daty i kwoty są zapisane na kilka sposobów.
Teraz zrobimy z tego porządną tabelę, na której da się liczyć; samo liczenie zrobimy w notebooku 07 (Pandas: analiza).

**Czego się nauczysz**
- zmienić nazwy kolumn, usunąć duplikaty i zbędne kolumny,
- naprawić teksty (spacje, wielkość liter),
- zamienić tekst na daty i na liczby,
- użyć własnej funkcji `clean_amount` na całej kolumnie (`apply`),
- zapisać czystą tabelę do nowego pliku Excela.

## 1. Wczytanie i plan czyszczenia

Czyszczenie to zawsze ta sama lista kroków, jak checklista przed złożeniem pisma. Nasz plan:

1. nazwy kolumn i zbędna kolumna `Źródło`,
2. duplikaty,
3. teksty: spacje i wielkość liter (gmina, wykonawca),
4. daty (`Data ogłoszenia`) i rok,
5. liczby: liczba ofert, wartość umowy, wolumen, okres umowy,
6. kody odpadów: ten sam znak między kodami, liczba kodów,
7. porządki, kontrola i zapis do `data/przetargi_clean.xlsx`.

Sam piszesz (z pomocą trenera) komórki w sekcjach 3 (Duplikaty), 4 (Teksty: spacje i wielkość liter),
7 (Kwoty i wolumen: nasza funkcja na całej kolumnie; samą funkcję tylko wklejasz) i 9 (Kody odpadów);
resztę tylko uruchamiasz i czytasz. Komórki z dopiskiem "(opcjonalnie, jeśli zostanie czas)" można pominąć
i uruchomić w domu.

In [ ]:
import pandas as pd   # pandas = biblioteka do tabel; skrót "pd" to przyjęty zwyczaj

# Wczytujemy surowy plik. df = DataFrame, czyli tabela jak arkusz w Excelu.
df = pd.read_excel("data/przetargi_2019_2024.xlsx")
print("Wiersze, kolumny na start:", df.shape)   # shape = (liczba wierszy, liczba kolumn)
df.head(3)                                        # pierwsze 3 wiersze do podglądu

## 2. Nazwy kolumn

`df["Wartość umowy"]` (wybór jednej kolumny po nazwie) działa, ale polskie znaki i spacje w nazwach
to proszenie się o literówki.
Zmienimy nazwy na krótkie `snake_case` (małe litery bez polskich znaków, słowa połączone podkreśleniem: `wartosc_umowy`)
przez słownik "stara nazwa -> nowa nazwa" (słownik = pary klucz: wartość w nawiasach klamrowych `{}`).
Kolumna `Źródło` (adres URL) nie jest potrzebna do analizy, więc ją usuniemy.

In [ ]:
# Słownik: klucz = nazwa DOKŁADNIE jak w Excelu, wartość = nowa nazwa.
# WIELKIE litery w nazwie zmiennej to umowa: "ustawienie, którego nie zmieniamy w trakcie".
COLUMN_NAMES = {
    "Numer ogłoszenia": "numer_ogloszenia",
    "Gmina": "gmina",
    "Województwo": "wojewodztwo",
    "Data ogłoszenia": "data_ogloszenia",
    "Tryb": "tryb",
    "Liczba ofert": "liczba_ofert",
    "Kody odpadów": "kody_odpadow",
    "Okres umowy": "okres_umowy",
    "Wartość umowy": "wartosc_umowy",
    "Wolumen odpadów": "wolumen_odpadow",
    "Wykonawca": "wykonawca",
    "Źródło": "zrodlo",
}

In [ ]:
# rename zwraca NOWĄ tabelę i nie rusza starej, dlatego zapisujemy wynik z powrotem do df.
# Pułapka: bez "df = " z przodu nic by się nie zmieniło.
df = df.rename(columns=COLUMN_NAMES)
df = df.drop(columns=["zrodlo"])   # drop = wyrzuć; columns=[...] = lista kolumn do wyrzucenia
# list() = zamień na zwykłą listę, żeby wynik był czytelny. W wyniku Python pokazuje teksty
# w pojedynczych cudzysłowach 'gmina'; to to samo, co nasze "gmina" w kodzie.
list(df.columns)                   # po: nowe nazwy

## 3. Duplikaty

Scrapowanie (automatyczne pobieranie ze stron) potrafi pobrać to samo ogłoszenie dwa razy. `duplicated()` mówi dla
każdego wiersza, czy identyczny wiersz już wystąpił wcześniej w tabeli (`True`/`False`), a `drop_duplicates()`
zostawia tylko pierwsze wystąpienie.

In [ ]:
# sum() na kolumnie True/False liczy, ile jest True, czyli ile wierszy to powtórki
print("Powtórzonych wierszy:", df.duplicated().sum())
print("Wierszy przed:", len(df))
df = df.drop_duplicates()          # drop_duplicates też zwraca nową tabelę, więc znów zapisujemy wynik do df
print("Wierszy po:", len(df))

## 4. Teksty: spacje i wielkość liter

Dla pandas `" Radom "`, `"radom"` i `"RADOM"` to trzy różne gminy. Operacje tekstowe `strip` (usuwa spacje z przodu
i z tyłu) i `title` (Pierwsza Litera Każdego Słowa Wielka) działają tak samo jak na pojedynczym tekście, tylko na
całej kolumnie, jeśli poprzedzisz je `.str.` (czytaj: "dla każdego tekstu w kolumnie").
Pułapka: `title()` robi wielką literę w każdym słowie (`Kostrzyn Nad Odrą`), ale do liczenia to nie przeszkadza.

In [ ]:
# Przed: pandas widzi dużo "różnych" gmin, a różnice to tylko spacje i wielkość liter.
# nunique() = ILE jest różnych wartości w kolumnie
print("Różnych nazw gmin przed:", df["gmina"].nunique())
print("Różnych nazw wykonawców przed:", df["wykonawca"].nunique())
# unique() = różne wartości, każda raz; sorted() układa je alfabetycznie (spacje z przodu lądują pierwsze);
# [:5] = pierwsze 5 elementów listy
print(sorted(df["gmina"].unique())[:5])
# .str.strip() = zetnij spacje z brzegów, .str.title() = Wielka Pierwsza Litera każdego słowa
df["gmina"] = df["gmina"].str.strip().str.title()
df["wykonawca"] = df["wykonawca"].str.strip()   # wykonawca: tylko spacje, wielkość liter zostawiamy
# Po: te same nazwy liczą się raz
print("Różnych nazw gmin po:", df["gmina"].nunique())
print("Różnych nazw wykonawców po:", df["wykonawca"].nunique())

## 5. Daty

`"15.03.2021"` to dla pandas tekst, a tekst sortuje się znak po znaku, więc `01.01.2023` wyląduje przed `27.01.2019`.
`pd.to_datetime` zamienia tekst na prawdziwą datę, a `format` mówi, jak go czytać:
`%d` dzień, `%m` miesiąc, `%Y` rok (4 cyfry).
Pułapka: wielkość liter ma znaczenie (`%M` to minuty, `%y` to rok dwucyfrowy).

In [ ]:
# Przed: sortujemy tekstowe daty. tolist() zamienia kolumnę na zwykłą listę, żeby łatwo ją podejrzeć.
print("Przed:", df.sort_values("data_ogloszenia")["data_ogloszenia"].head(3).tolist())
# Zamiana tekstu na datę według wzoru dzień.miesiąc.rok
df["data_ogloszenia"] = pd.to_datetime(df["data_ogloszenia"], format="%d.%m.%Y")
df["rok"] = df["data_ogloszenia"].dt.year   # .dt.year wyciąga rok z daty; nowa kolumna "rok" przyda się do liczenia per rok
# Po: sortowanie działa jak w kalendarzu
df.sort_values("data_ogloszenia")[["numer_ogloszenia", "gmina", "data_ogloszenia", "rok"]].head(3)

## 6. Liczba ofert: tekst na liczbę całkowitą

`"2"` to tekst, więc pandas nie policzy z niego średniej. `pd.to_numeric` zamienia tekst na liczby,
a `errors="coerce"` znaczy: czego nie da się zamienić (np. `"brak danych"`), zostaje puste
(puste pole pandas pokazuje jako `NaN` albo `<NA>`).
`astype("Int64")` (duże I) = liczby całkowite, które tolerują puste pola;
zwykłe `int` wywróciłoby się na pierwszym braku.

In [ ]:
# Najpierw "na sucho": wynik do osobnej zmiennej, żeby zobaczyć, które wiersze staną się puste (df nie ruszamy)
offers_numeric = pd.to_numeric(df["liczba_ofert"], errors="coerce")
# isna() = czy pole puste (True/False). df.loc[warunek, lista kolumn] = filtr wierszy i wybór kolumn naraz
# (przed przecinkiem wiersze, po przecinku kolumny); zapamiętujemy te wiersze z oryginalnym tekstem
became_empty = df.loc[offers_numeric.isna(), ["numer_ogloszenia", "gmina", "liczba_ofert"]]
print("Pustych pól po zamianie:", len(became_empty))
# Teraz na serio: zapisujemy do df i od razu ustawiamy typ całkowity z dopuszczalnymi brakami
df["liczba_ofert"] = offers_numeric.astype("Int64")
print("Typ kolumny:", df["liczba_ofert"].dtype)   # dtype = typ kolumny; Int64 zamiast tekstu
became_empty.head()   # co było w oryginale: "brak danych" albo nic (NaN)

## 7. Kwoty i wolumen: nasza funkcja na całej kolumnie

Kwoty są zapisane na cztery sposoby (`1 234 567,89 zł`, `... PLN`, bez spacji, z twardą spacją), a `clean_amount`,
nasza funkcja z notebooku 04 (Funkcje), czyści JEDEN taki tekst. `apply(clean_amount)` znaczy:
"wywołaj tę funkcję dla każdej komórki w kolumnie".
Pułapka: bez nawiasów po nazwie funkcji, bo podajemy przepis, a nie jego wynik.

In [ ]:
# Nasza clean_amount, wklejona bez zmian (każdy notebook startuje od zera, więc funkcję trzeba zdefiniować tu jeszcze raz).
# Zostawia z tekstu cyfry, przecinek i kropkę, zamienia przecinek na kropkę i robi z tego liczbę.
# Gdy nie ma żadnej cyfry, zwraca None (brak wartości).
# Tekst w potrójnych cudzysłowach pod def to opis dla człowieka; Python go pomija.
def clean_amount(text):
    """Zamienia tekst w stylu '1 234 567,89 zł' na liczbę 1234567.89. Gdy nie ma cyfr -> None."""
    digits = ""
    for char in str(text):
        if char.isdigit() or char in ",.":
            digits = digits + char
    if digits == "":
        return None
    return float(digits.replace(",", "."))

# Szybki test na dwóch zapisach z pliku
print(clean_amount("14 737 955,32 zł"))
print(clean_amount("12 780 445,51 PLN"))

In [ ]:
# apply(clean_amount) = dla każdej komórki wywołaj clean_amount i zbierz wyniki w nową kolumnę
df["wartosc_pln"] = df["wartosc_umowy"].apply(clean_amount)
df["wolumen_mg"] = df["wolumen_odpadow"].apply(clean_amount)
# Przed/po obok siebie: stary tekst i nowa liczba (puste pole zostaje puste, NaN)
df[["wartosc_umowy", "wartosc_pln", "wolumen_odpadow", "wolumen_mg"]].head(3)

**Niewidoczny wróg: twarda spacja (opcjonalnie, jeśli zostanie czas)**

Twarda spacja `\xa0` wygląda jak spacja, ale nią nie jest (częsta pamiątka po scrapowaniu). Naszej funkcji to nie
przeszkadza, bo zostawia same cyfry; sprawdźmy.

In [ ]:
# .str.contains("\xa0") = czy tekst zawiera twardą spację; na=False: puste pole traktuj jako "nie zawiera"
has_nbsp = df["wartosc_umowy"].str.contains("\xa0", na=False)
print("Wierszy z twardą spacją:", has_nbsp.sum())
# repr() pokazuje tekst "od kuchni", z ukrytymi znakami; iloc[0] = pierwszy pasujący wiersz
print(repr(df.loc[has_nbsp, "wartosc_umowy"].iloc[0]))
# Wiersze z twardą spacją też stały się liczbami
df.loc[has_nbsp, ["wartosc_umowy", "wartosc_pln", "wolumen_odpadow", "wolumen_mg"]].head(3)

## 8. Okres umowy

Okres to tekst w dwóch zapisach (`24 miesiące` i `24 mies.`), a potrzebujemy samej liczby miesięcy.
Ten sam przepis co dla kwot.

In [ ]:
# Przed: wszystkie zapisy okresu (sorted() układa je alfabetycznie, więc pary stoją obok siebie)
print(sorted(df["okres_umowy"].unique()))
# apply(clean_amount) daje liczby z przecinkiem (24.0); astype(int) robi z nich całkowite (24).
# Tu nie ma pustych pól, więc zwykły int wystarczy (przy brakach musiałby być "Int64", jak przy liczbie ofert).
df["okres_mies"] = df["okres_umowy"].apply(clean_amount).astype(int)
# Po: value_counts() = ile razy występuje każda wartość; 5 wartości zamiast 10 zapisów.
# sort_index() układa wynik po wartości (12, 18, 24...), a nie po liczności
df["okres_mies"].value_counts().sort_index()

## 9. Kody odpadów

Kody są rozdzielone raz `; `, raz `, `. Ujednolicamy to przez `.str.replace` (podmień fragment tekstu na inny,
w każdej komórce), a liczbę kodów liczymy jako liczbę średników + 1.

In [ ]:
# Przed: ile wierszy ma przecinek między kodami?
# .str.contains(", ") = czy tekst zawiera przecinek ze spacją (True/False dla każdego wiersza); sum() liczy True
print("Z przecinkiem przed:", df["kody_odpadow"].str.contains(", ").sum())
df["kody_odpadow"] = df["kody_odpadow"].str.replace(", ", "; ")   # .str.replace(co, na_co) w każdej komórce
print("Z przecinkiem po:", df["kody_odpadow"].str.contains(", ").sum())
# Liczba kodów = liczba średników + 1 (trzy kody mają dwa średniki); .str.count liczy wystąpienia znaku
df["liczba_kodow"] = df["kody_odpadow"].str.count(";") + 1
df[["kody_odpadow", "liczba_kodow"]].head()

**Twoja kolej:** jaki udział przetargów obejmuje kod `20 03 01` (odpady zmieszane)?

In [ ]:
# contains daje True/False dla każdego wiersza; mean() z True/False to udział (True liczy się jako 1, False jako 0)
share_mixed = df["kody_odpadow"].str.contains("20 03 01").mean()
# TODO: wypisz share_mixed jako procent f-stringiem: print(f"...{share_mixed:.1%}"); :.1% = procent z 1 miejscem po przecinku.
# Potem podmień kod na "15 01 01" (papier i tektura).
print(share_mixed)

## 10. Porządki i kontrola

Sekcje 10 (Porządki i kontrola) i 11 (Zapis czystej tabeli) tylko uruchamiamy i czytamy. Stare kolumny tekstowe mają
już liczbowe odpowiedniki, więc je usuwamy, sortujemy po dacie i numerujemy wiersze od nowa. Na koniec trzy kontrole,
które warto robić po każdym czyszczeniu: typy (`info`), braki (`isna`), statystyki (`describe`).

In [ ]:
df = df.drop(columns=["okres_umowy", "wartosc_umowy", "wolumen_odpadow"])   # zbędne już kolumny tekstowe
# Wybór kolumn listą nazw (podwójne nawiasy) ustawia też ich kolejność: okres przed kwotami;
# w tej kolejności zapiszemy czysty plik
df = df[["numer_ogloszenia", "gmina", "wojewodztwo", "data_ogloszenia", "tryb", "liczba_ofert", "kody_odpadow",
         "wykonawca", "rok", "okres_mies", "wartosc_pln", "wolumen_mg", "liczba_kodow"]]
# sort_values układa po dacie; reset_index(drop=True) numeruje wiersze od 0 i wyrzuca stare numery
df = df.sort_values("data_ogloszenia").reset_index(drop=True)
df.head()

**Trzy kontrole po czyszczeniu (opcjonalnie, jeśli zostanie czas)**

In [ ]:
# Kontrola 1: typy kolumn (Dtype). Daty i liczby zamiast samego tekstu (tekst to "str" lub "object", zależy od wersji pandas).
# "RangeIndex: 456 entries" = 456 wierszy ponumerowanych od 0; "Non-Null Count" = ile pól jest wypełnionych.
df.info()
print()   # pusta linia, żeby oddzielić kontrole w wyniku
# Kontrola 2: liczba pustych pól w każdej kolumnie. Braki są normalne; ważne, żeby wiedzieć, gdzie są.
print("Puste pola w każdej kolumnie:")
print(df.isna().sum())
print()
# Kontrola 3: describe() podsumowuje tylko kolumny liczbowe. Skoro kwoty, wolumen i okres są już liczbami,
# powinny się tu pojawić (wybieramy je listą nazw). Wiersze: count = ile pól wypełnionych, mean = średnia,
# std = jak bardzo wartości się rozrzucają, min/max = najmniejsza/największa, 50% = mediana (połowa wartości jest niżej),
# 25% i 75% = jedna czwarta i trzy czwarte wartości jest niżej.
# round(1) = jedno miejsce po przecinku; kwoty są w złotych (nie w milionach), więc liczby są długie.
df[["liczba_ofert", "okres_mies", "wartosc_pln", "wolumen_mg", "liczba_kodow"]].describe().round(1)

## 11. Zapis czystej tabeli

`to_excel` zapisuje tabelę do nowego pliku; `index=False` = bez kolumny z numerami wierszy.
Pułapka: jeśli ten plik jest otwarty w Excelu, zapis się nie uda (zamknij plik i uruchom komórkę ponownie).
Excel nie zna typu `Int64`, więc po ponownym wczytaniu tego pliku `liczba_ofert` wróci jako liczba z przecinkiem
(`1.0` zamiast `1`); wtedy wystarczy jeszcze raz `astype("Int64")`.

In [ ]:
# Zapisujemy czystą tabelę do nowego pliku; surowy plik zostaje nietknięty.
df.to_excel("data/przetargi_clean.xlsx", index=False)
print("Zapisano data/przetargi_clean.xlsx:", len(df), "wierszy,", len(df.columns), "kolumn")

## 12. Jak podmienić na swoje dane

Ten notebook jest gotowym przepisem. Na własnym pliku zmieniasz tylko trzy rzeczy:

1. **ścieżkę** w `pd.read_excel(...)` w sekcji 1 (Wczytanie i plan czyszczenia); jeśli dane są w innym arkuszu,
   dodaj `sheet_name="nazwa arkusza"`,
2. **słownik `COLUMN_NAMES`** w sekcji 2 (Nazwy kolumn): klucze = Twoje nagłówki z Excela, dokładnie jak są;
   wartości zostaw bez zmian,
3. **format daty** w `pd.to_datetime` w sekcji 5 (Daty), np. `"%Y-%m-%d"` dla `2021-03-15`.

Reszta kodu nie wymaga zmian, o ile kolumny znaczą to samo (wartość = cała umowa brutto,
wolumen = cały okres w Mg, czyli w tonach).
Po wczytaniu zawsze sprawdź: `df.head()` (czy to ten plik), `df.info()` (typy), `df.isna().sum()` (braki),
a po czyszczeniu `describe()` (czy liczby mają sens: minimum, maksimum, mediana).

## Zadania

Jeśli brakuje czasu, zadania zostają na pracę domową. Rozwiązania są na końcu notebooka.

1. Ile różnych gmin jest w surowym pliku przed czyszczeniem, a ile po `strip()` i `title()`?
   Użyj świeżo wczytanego surowego pliku.
2. Pokaż wiersze, w których brakuje wartości umowy (`wartosc_pln`). Ile ich jest?
3. Dodaj kolumnę `ma_zmieszane`: `True`, gdy kody odpadów zawierają `20 03 01`. Ile jest `True`, a ile `False`?
4. Ile przetargów w 2023 roku miało jedną ofertę i jaki to udział wszystkich przetargów z 2023?
   (filtr z dwóch warunków + `len`). Potem ten sam schemat dla trybu podstawowego w 2023
   (kolumna `tryb`, wartość `"tryb podstawowy"`).
5. (bonus) Na surowym pliku znajdź wiersze, w których okres umowy ma skrót `mies.`.
   Użyj `.str.endswith("mies.")` (sprawdza końcówkę tekstu), nie `contains`.

In [ ]:
# Zadanie 1
# TODO: policz nunique() kolumny "Gmina" w df_raw przed i po .str.strip().str.title()
df_raw = pd.read_excel("data/przetargi_2019_2024.xlsx")   # świeży surowy plik, z oryginalnymi nazwami kolumn
# Podpowiedź: print("Przed:", df_raw["Gmina"].nunique())

In [ ]:
# Zadanie 2
# TODO: filtr df[df["wartosc_pln"].isna()] wybiera wiersze z pustą wartością; len() policzy, ile ich jest
# Podpowiedź: zapisz filtr do zmiennej missing_value i pokaż kilka kolumn, np. numer, gminę, rok

In [ ]:
# Zadanie 3
# TODO: df["ma_zmieszane"] = ... .str.contains("20 03 01"); potem value_counts() (ile True, ile False) na nowej kolumnie
# Podpowiedź: nowa kolumna powstaje przez zwykłe przypisanie df["nazwa"] = ...

In [ ]:
# Zadanie 4
# TODO 1: mask_single = (df["rok"] == 2023) & (df["liczba_ofert"] == 1); każdy warunek w nawiasach, łączymy przez &
# Podpowiedź: share_single = len(df[mask_single]) / len(df[df["rok"] == 2023]); wypisz f-stringiem z :.1%
# TODO 2: ten sam schemat dla mask_basic = (df["rok"] == 2023) & (df["tryb"] == "tryb podstawowy")
# Podpowiedź: w danych są dwa tryby ("przetarg nieograniczony" i "tryb podstawowy"); ten drugi występuje od 2021

In [ ]:
# Zadanie 5 (bonus)
# TODO: df_raw["Okres umowy"].str.endswith("mies.") daje True dla skrótu; pokaż te wiersze przez df_raw[...]
# Podpowiedź: .str.endswith("mies.") = czy tekst kończy się na "mies."
# (sprawdza końcówkę każdego tekstu w kolumnie)

## Rozwiązania

In [ ]:
# Rozwiązanie 1: spacje i wielkość liter mnożą "różne" gminy; po czyszczeniu zostaje prawdziwa liczba
df_raw = pd.read_excel("data/przetargi_2019_2024.xlsx")
print("Gmin przed czyszczeniem:", df_raw["Gmina"].nunique())
print("Gmin po czyszczeniu:", df_raw["Gmina"].str.strip().str.title().nunique())
print("Gmin w naszej czystej tabeli:", df["gmina"].nunique())   # ta sama liczba, bo to ten sam przepis

In [ ]:
# Rozwiązanie 2: isna() daje True tam, gdzie pole jest puste; filtr zostawia tylko takie wiersze
missing_value = df[df["wartosc_pln"].isna()]
print("Przetargów bez wartości umowy:", len(missing_value))
missing_value[["numer_ogloszenia", "gmina", "rok", "wartosc_pln", "wolumen_mg"]]

In [ ]:
# Rozwiązanie 3: contains daje True/False dla każdego przetargu; przypisanie tworzy nową kolumnę
df["ma_zmieszane"] = df["kody_odpadow"].str.contains("20 03 01")
df["ma_zmieszane"].value_counts()   # w naszych danych każdy przetarg obejmuje zmieszane, więc same True

In [ ]:
# Rozwiązanie 4: dwa warunki, każdy w nawiasach, połączone & (a nie "and", bo porównujemy całe kolumny)
mask_single = (df["rok"] == 2023) & (df["liczba_ofert"] == 1)
count_2023 = len(df[df["rok"] == 2023])   # mianownik: wszystkie przetargi z 2023
count_single = len(df[mask_single])
share_single = count_single / count_2023
print(f"Jedna oferta w 2023: {count_single} z {count_2023}, czyli {share_single:.1%}")
# Ten sam schemat dla trybu podstawowego. W danych są dwa tryby: "przetarg nieograniczony" i "tryb podstawowy"
# (ten drugi od 2021, dla mniejszych umów). Na prawdziwych danych kod będzie ten sam, tylko nazwy trybów inne.
mask_basic = (df["rok"] == 2023) & (df["tryb"] == "tryb podstawowy")
count_basic = len(df[mask_basic])
print(f"Tryb podstawowy w 2023: {count_basic} z {count_2023}, czyli {count_basic / count_2023:.1%}")

In [ ]:
# Rozwiązanie 5 (bonus): .str.endswith("mies.") = czy tekst kończy się na skrót "mies."
# (True/False dla każdego wiersza)
has_abbrev = df_raw["Okres umowy"].str.endswith("mies.")   # df_raw = surowy plik wczytany w Rozwiązaniu 1
print("Wierszy ze skrótem 'mies.':", has_abbrev.sum())
df_raw.loc[has_abbrev, ["Numer ogłoszenia", "Gmina", "Okres umowy"]].head()

**Podsumowanie**

- Czyszczenie to lista powtarzalnych kroków: nazwy kolumn, duplikaty, teksty, daty, liczby, kontrola, zapis.
- Tekst zamieniamy na liczby przez `pd.to_numeric` (proste przypadki) albo własną funkcję i `apply` (brudne kwoty).
- Po każdym kroku sprawdzamy "przed/po": `nunique()`, `isna().sum()`, `info()`, `describe()`.
- Wynik: `data/przetargi_clean.xlsx`, 456 przetargów, na których w notebooku 07 (Pandas: analiza) policzymy
  ceny, udziały i trendy.

Jeśli na zajęciach pominęliśmy komórki "(opcjonalnie, jeśli zostanie czas)", uruchom je w domu i przeczytaj wyniki:
twardą spację w sekcji 7 (Kwoty i wolumen: nasza funkcja na całej kolumnie) i trzy kontrole w sekcji 10 (Porządki i kontrola).
Jeśli zabrakło czasu na Zadania, zrób je w domu i porównaj z Rozwiązaniami.

Dalej: notebook 07 (Pandas: analiza), plik `07_pandas_analiza.ipynb`